# Praktikum Kecerdasan Artifisial Lanjut


---
## Bab 6. Support Vector Machine (SVM)


Tuliskan Nama, NIM, dan kelas Anda:

Nama  : Hassan Nashrallah

NIM   : 245150200111025

Kelas : F

### 1) Import Data

Unduh dataset yang akan digunakan pada praktikum kali ini. Anda dapat menggunakan aplikasi wget untuk mendowload dataset dan menyimpannya dalam Google Colab. Jalankan cell di bawah ini untuk mengunduh dataset

In [ ]:
! wget https://gist.githubusercontent.com/Thanatoz-1/9e7fdfb8189f0cdf5d73a494e4a6392a/raw -O iris.csv

--2026-04-15 23:42:16--  https://gist.githubusercontent.com/Thanatoz-1/9e7fdfb8189f0cdf5d73a494e4a6392a/raw
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.108.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4626 (4.5K) [text/plain]
Saving to: ‘iris.csv’

iris.csv            100%[===================>]   4.52K  --.-KB/s    in 0s      

2026-04-15 23:42:16 (68.7 MB/s) - ‘iris.csv’ saved [4626/4626]



Setelah dataset berhasil diunduh, langkah berikutnya adalah membaca dataset dengan memanfaatkan fungsi **readcsv** dari library pandas. Lakukan pembacaan berkas csv ke dalam dataframe dengan nama **data** menggunakan fungsi **readcsv**. Jangan lupa untuk melakukan import library pandas terlebih dahulu


In [2]:
import numpy as np
import pandas as pd

data = pd.read_csv('iris.csv')



Cek isi dataset Anda dengan menggunakan perintah **head()**

In [3]:
data.drop(data[data['target']=='Iris-virginica'].index, inplace=True)
data['target'] = data['target'].map({'Iris-setosa': -1, 'Iris-versicolor': 1})
data.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,-1
1,4.9,3.0,1.4,0.2,-1
2,4.7,3.2,1.3,0.2,-1
3,4.6,3.1,1.5,0.2,-1
4,5.0,3.6,1.4,0.2,-1


## 2) Membagi data menjadi data latih dan data uji

Metode pembelajaran mesin memerlukan dua jenis data :


1.   Data latih : Digunakan untuk proses training metode klasifikasi
2.   Data uji : Digunakan untuk proses evaluasi metode klasifikasi

Data uji dan data latih perlu dibuat terpisah (mutualy exclusive) agar hasil evaluasi lebih akurat.

Data uji dan data latih dapat dibuat dengan cara membagi dataset dengan rasio tertentu, misalnya 80% data latih dan 20% data uji.

Library Scikit-learn memiliki fungsi [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) pada modul **model_selection** untuk membagi dataset menjadi data latih dan data uji. Bagilah dataset anda menjadi dua, yaitu **data_latih** dan **data_uji**.


In [4]:
from sklearn.model_selection import train_test_split
data_latih, data_uji = train_test_split(data,test_size=0.2)

Tampilkan banyaknya data pada **data_latih** dan **data_uji**. Seharusnya **data_latih** terdiri dari 120 data, dan **data_uji** terdiri dari 30 data

In [5]:
print(data_latih.shape[0])
print(data_uji.shape[0])

80
20


Pisahkan label/kelas dari data uji menjadi sebuah variabel bernama **label_uji**

In [9]:
label_latih = data_latih.pop('target')
label_uji = data_uji.pop('target')

## 3) Pembentukan data latih one-vs-rest

Metode one-vs-rest memerlukan tiga jenis data latih yang diperlukan untuk melatih tiga SVM yang berbeda pada dataset Iris. Fungsi **buat_trainingset** digunakan untuk membentuk tiga dataset tersebut.

In [6]:
def buat_trainingset(dataset):
    trainingset = {}
    kolom_kelas = dataset.columns[-1]
    list_kelas = dataset[kolom_kelas].unique()
    for kelas in list_kelas:
        data_temp = dataset.copy(deep=True)
        data_temp[kolom_kelas]=data_temp[kolom_kelas].map({kelas:1})
        data_temp[kolom_kelas]=data_temp[kolom_kelas].fillna(-1)
        trainingset[kelas]=data_temp
    return trainingset

Gunakan fungsi **buat_trainingset** untuk membentuk data latih dengan nama variabel **trainingset** yang akan digunakan pada proses training.

In [7]:
trainingset = buat_trainingset(data_latih)

Tampilkan isi **trainingset** agar Anda dapat memahami struktur dari variabel tersebut.

In [8]:
print(trainingset)

{np.int64(1):     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
54                6.5               2.8                4.6               1.5   
6                 4.6               3.4                1.4               0.3   
22                4.6               3.6                1.0               0.2   
8                 4.4               2.9                1.4               0.2   
20                5.4               3.4                1.7               0.2   
..                ...               ...                ...               ...   
58                6.6               2.9                4.6               1.3   
72                6.3               2.5                4.9               1.5   
3                 4.6               3.1                1.5               0.2   
93                5.0               2.3                3.3               1.0   
23                5.1               3.3                1.7               0.5   

    target  
54     1.0  

## 4) Pembentukan SVM Biner

Tujuan dari algoritma SVM adalah meminimalkan nilai *cost function*. Penghitungan nilai minimal dapat dapat dilakukan dengan menghitung nilai gradien dari *cost function* terlebih dahulu. Fungsi di bawah ini berguna untuk menghitung nilai gradien cost function

In [10]:
def hitung_cost_gradient(W,X,Y,regularization):
    jarak = 1 - (Y* np.dot(X,W))
    dw = np.zeros(len(W))
    if max(0,jarak) == 0:
        di = W
    else:
        di = W - (regularization * Y * X)
    dw += di
    return dw

Terdapat beberapa cara untuk meminimalkan nilai *cost function*, salah satunya menggunakan Stochastic Gradient Descent (SGD) untuk melakukan minimasi. Minimasi *cost function* merupakan inti dari algoritma SVM. Fungsi di bawah ini merupakan implementasi algoritma SGD

In [11]:
from sklearn.utils import shuffle

def sgd(data_latih, label_latih, learning_rate=0.000001, max_epoch=1000, regularization=10000):
    data_latih = data_latih. to_numpy ()
    label_latih = label_latih.to_numpy()
    bobot = np.zeros(data_latih.shape[1])
    for epoch in range(1,max_epoch):
        X,Y = shuffle(data_latih,label_latih, random_state=101)
        for index, x in enumerate(X):
            delta = hitung_cost_gradient(bobot, x, Y[index], regularization)
            bobot = bobot - (learning_rate * delta)
    return bobot

## 5) Proses Training

Proses training dilakukan dengan memanggil fungsi **sgd** berulang kali sesuai banyaknya kelas yang ada pada data. Dengan demikian, proses training menghasilkan bobot sebanyak kelas yang ada pada dataset. Buatlah fungsi bernama **training** yang digunakan untuk melakukan proses training one-vs-rest

In [12]:
def training(trainingset):
    list_kelas = trainingset.keys()
    w = {}
    for kelas in list_kelas:
        data_latih = trainingset[kelas]
        label_latih = data_latih.pop(data_latih.columns[-1])
        w[kelas] = sgd(data_latih, label_latih)
    return w

Lakukan proses training dengan memanggil fungsi **training** dan menempatkan hasilnya pada variabel **W**

In [13]:
W = training(trainingset)

Tampilkan isi variabel **W**

In [14]:
print(W)

{np.int64(1): array([-0.2726977 , -0.66899244,  1.16290723,  0.53591549]), np.int64(-1): array([ 0.2726977 ,  0.66899244, -1.16290723, -0.53591549])}


## 6) Proses *testing* biner
Proses testing dilakukan dengan menghitung nilai [*dot product*](https://en.wikipedia.org/wiki/Dot_product) antara bobot hasil training dengan data uji. Kelas data ditentukan berdasarkan tanda (positif atau negatif) dari hasil dot product tersebut. Fungsi berikut mengimplementasikan proses testing

In [15]:
def testing(W, data_uji):
    prediksi = []
    
    for i in range(data_uji.shape[0]):
        x = data_uji.to_numpy()[i]
        skor = {kelas: np.dot(bobot, x) for kelas, bobot in W.items()}        
        kelas_prediksi = max(skor, key=skor.get)
        prediksi.append(kelas_prediksi)
    return np.array(prediksi)

In [16]:
y_prediksi = testing(W, data_uji)
print(sum(y_prediksi == label_uji))

20


## Tugas
Pada tugas kali ini Anda mendefinisikan proses testing pada metode one-vs-rest. Proses testing pada metode one-vs-rest dilakukan dengan memanggil proses testing biner untuk setiap **value** pada dictionary **W**. Kelas pada sebuah data latih adalah **key** pada dictionary **W** yang memiliki nilai prediksi **1**. Lengkapi fungsi **testing_onevsrest** di bawah ini. Output dari fungsi tersebut adalah list nama kelas hasil prediksi.

In [17]:
data_tugas = pd.read_csv('iris.csv')

In [18]:
data_latih_tugas, data_uji_tugas = train_test_split(data_tugas, test_size=0.2)

print(data_uji_tugas.shape[0])
print(data_latih_tugas.shape[0])

label_uji_tugas = data_uji_tugas.pop('target')

30
120


In [19]:
def buat_trainingset(dataset):
    trainingset = {}
    kolom_kelas = dataset.columns[-1]
    list_kelas = dataset[kolom_kelas].unique()

    for kelas in list_kelas:
        data_temp = dataset.copy(deep=True)
        data_temp[kolom_kelas] = data_temp[kolom_kelas].map({kelas: 1})
        data_temp[kolom_kelas] = data_temp[kolom_kelas].fillna(-1)
        trainingset[kelas] = data_temp

    return trainingset

In [20]:
trainingset = buat_trainingset(data_latih_tugas)
print(trainingset)

{'Iris-virginica':      sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
145                6.7               3.0                5.2               2.3   
48                 5.3               3.7                1.5               0.2   
139                6.9               3.1                5.4               2.1   
101                5.8               2.7                5.1               1.9   
62                 6.0               2.2                4.0               1.0   
..                 ...               ...                ...               ...   
82                 5.8               2.7                3.9               1.2   
120                6.9               3.2                5.7               2.3   
10                 5.4               3.7                1.5               0.2   
61                 5.9               3.0                4.2               1.5   
98                 5.1               2.5                3.0               1.1   

     tar

In [21]:
def training(trainingset):
    list_kelas = trainingset.keys()
    W = {}

    for kelas in list_kelas:
        data_latih = trainingset[kelas]
        label_latih = data_latih.pop(data_latih.columns[-1])
        W[kelas] = sgd(data_latih, label_latih)

    return W

In [22]:
W = training(trainingset)
print(W)

{'Iris-virginica': array([-3.12896698, -3.74496033,  4.33137404,  5.29377007]), 'Iris-setosa': array([ 0.16125231,  0.67852275, -1.01360136, -0.46193569]), 'Iris-versicolor': array([ 0.91438105, -2.18498016,  1.22805786, -3.51651851])}


In [23]:
def testing_onevsrest(W,data_uji):
    kelas_prediksi = []
    for i in range(data_uji.shape[0]):
        x = data_uji.to_numpy()[i]
        hasil = {}
        for kelas, bobot in W.items():
            hasil[kelas] = np.sign(np.dot(bobot, x))
        kandidat = [kelas for kelas, nilai in hasil.items() if nilai == 1]
        if len(kandidat) == 1:
            kelas_prediksi.append(kandidat[0])
        else:
            skor = {kelas: np.dot(bobot, x) for kelas, bobot in W.items()}
            kelas_prediksi.append(max(skor, key=skor.get))
    return kelas_prediksi

#### Penjelasan

Fungsi testing_onevsrest(W, data_uji) digunakan untuk melakukan proses pengujian pada model klasifikasi dengan pendekatan One-vs-Rest. Parameter W merepresentasikan kumpulan bobot untuk setiap kelas yang diperoleh dari proses pelatihan, sedangkan data_uji merupakan data yang akan diprediksi kelasnya. Proses dimulai dengan melakukan iterasi terhadap setiap data uji. Untuk setiap data x, dihitung nilai dot product antara vektor fitur dan bobot masing-masing kelas, lalu diterapkan sign function. Hasil ini digunakan untuk menentukan apakah suatu data termasuk ke dalam kelas tertentu (nilai +1) atau tidak (nilai -1).

Selanjutnya, dilakukan identifikasi kelas-kelas yang menghasilkan nilai +1 sebagai kandidat prediksi. Jika hanya terdapat satu kandidat, maka kelas tersebut langsung ditetapkan sebagai hasil prediksi. Namun, apabila tidak terdapat kandidat atau terdapat lebih dari satu kandidat, maka dilakukan perhitungan ulang menggunakan nilai dot product tanpa sign function untuk seluruh kelas. Kelas dengan nilai skor tertinggi dipilih sebagai hasil prediksi akhir. Fungsi ini mengembalikan daftar kelas prediksi untuk seluruh data uji.

#### Berapa banyak data latih yang berhasil diprediksi dengan benar?

In [24]:
prediksi = testing_onevsrest(W, data_uji_tugas)
print(sum(prediksi == label_uji_tugas))

28


Dari kode tersebut, didapatkan 28 dari 30 data latih berhasil diprediksi dengan benar testing one-vs-rest. Rasio akurasi testing one-vs-rest lebih rendah jika dibandingkan dengan testing biner yang berhasil memprediksi seluruh 20 data latih dengan benar.